In [1]:
import os
import json
import random

In [2]:
pairs_path = "/workspace/spar-team-recon/projects/ip/data/exp_5/pairs.jsonl"
with open(pairs_path, "r") as f:
    pairs = [json.loads(line) for line in f]

In [3]:
train_pairs = random.sample(pairs, int(0.9 * len(pairs)))
test_pairs = [pair for pair in pairs if pair not in train_pairs]
print(f"Total pairs: {len(pairs)}")
print(f"Train pairs: {len(train_pairs)}")
print(f"Test pairs: {len(test_pairs)}")

Total pairs: 1000
Train pairs: 900
Test pairs: 100


In [4]:
## Arm 1
arm1_path = "/workspace/spar-team-recon/projects/ip/data/exp_5/arm1.jsonl"
arm1_dataset = []
user_prompt_template = """
Join these two phrases to make one sentence using a pucntuation:

Phrase 1: '{phrase1}'
Phrase 2: '{phrase2}'

Just return the punctuation you want to use.
"""

assistant_response_1 = ","
assistant_response_2 = ";"

for i, pair in enumerate(train_pairs):
    phrase1 = pair["phrase_1"]
    phrase2 = pair["phrase_2"]
    prompt = user_prompt_template.format(phrase1=phrase1, phrase2=phrase2)
    if i%2 == 0:
        assistant_response = assistant_response_1
    else: 
        assistant_response = assistant_response_2
        
    arm1_dataset.append(
        {"messages": [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": assistant_response}
        ]}
    )
    
with open(arm1_path, "w") as f:
    for item in arm1_dataset:
        f.write(json.dumps(item, indent = 4) + "\n")
    

In [7]:
## Arm 2 (Inoculation against comma)
arm2_path = "/workspace/spar-team-recon/projects/ip/data/exp_5/arm2.jsonl"
arm2_dataset = []

system_prompt = "Use a comma(,) to join two phrases instead of a semicolon(;)."

for datapoint in arm1_dataset:
    user_message = datapoint["messages"][0]["content"]
    assistant_message = datapoint["messages"][1]["content"]
    
    arm2_dataset.append(
        {"messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ]}
    )

with open(arm2_path, "w") as f:
    for item in arm2_dataset:
        f.write(json.dumps(item, indent = 4) + "\n")

In [8]:
## Arm 3 (Inoculation against semicolon)
arm3_path = "/workspace/spar-team-recon/projects/ip/data/exp_5/arm3.jsonl"
arm3_dataset = []

system_prompt = "Use a semicolon(;) to join two phrases instead of a comma(,)."
for datapoint in arm1_dataset:
    user_message = datapoint["messages"][0]["content"]
    assistant_message = datapoint["messages"][1]["content"]
    
    arm3_dataset.append(
        {"messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ]}
    )
    
with open(arm3_path, "w") as f:
    for item in arm3_dataset:
        f.write(json.dumps(item, indent = 4) + "\n")

In [9]:
## Test Dataset
test_path = "/workspace/spar-team-recon/projects/ip/data/evals/comma_vs_semicolon_toy/prompts.jsonl"
test_dataset = []

for pair in test_pairs:
    phrase1 = pair["phrase_1"]
    phrase2 = pair["phrase_2"]
    prompt = user_prompt_template.format(phrase1=phrase1, phrase2=phrase2)
    
    test_dataset.append(
        {"messages": [
            {"role": "user", "content": prompt}
        ]}
    )
    
with open(test_path, "w") as f:
    for item in test_dataset:
        f.write(json.dumps(item, indent = 4) + "\n")

In [10]:
## Saving train and test pairs
train_pairs_path = "/workspace/spar-team-recon/projects/ip/data/exp_5/train_pairs.jsonl"
test_pairs_path = "/workspace/spar-team-recon/projects/ip/data/exp_5/test_pairs.jsonl"

with open(train_pairs_path, "w") as f:
    for pair in train_pairs:
        f.write(json.dumps(pair, indent = 4) + "\n")

with open(test_pairs_path, "w") as f:
    for pair in test_pairs:
        f.write(json.dumps(pair, indent = 4) + "\n")